### Word2Vec
- 문자를 수치형으로 변환 시켜주는 딥러닝 기반의 임베딩 기술
- 매개변수
    - sentences
        - 기본값 : None
        - 토큰화가 된 문장 데이터 (2차원 데이터)
        - None 기본값 ?? -> 학습을 시킬수 있다.
    - vector_size
        - 기본값 : 100
        - 임베딩 벡터 차원의 개수 ( feature의 수 )
    - window
        - 기본값 : 5
        - 예측 시 고려할 주변 단어와의 거리 (문맥의 크기)
    - sg
        - 기본값 : 0
        - 0인 경우
            - CBOW 방식 ( 주변 단어들을 이용하여 중심 단어를 예측 )
        - 1인 경우
            - Skip_gram 방식 (중심 단어를 이용하여 주변 단어를 예측)
        - 빠른 계산이 필요한 경우라면 0을 사용
        - 일반적으로는 1을 사용
    - min_count
        - 기본값 : 5
        - 최소 등장 빈도 수
        - 적게 등장한 단어들을 제외
    - hs
        - 기본값 : 0
        - 계산의 방식 지정
        - 0 : Negative Sampling (계산량이 적음)
        - 1 : Hierarchical Softmax (계산량 많음)
    - epochs
        - 기본값 : 100  
        - 반복 학습 횟수 지정
    - max_vocab_size
        - 기본값 : None
        - 메모리 제한시 사용할 최대 단어의 개수
- 속성
    - wv
        - 학습된 단어 벡터 (class 형태로 출력)
        - 예 : model.wv['단어']
    - wv.index_to_key
        - 단어의 리스트(학습이 된 단어의 개수) -> 최소 등장 횟수에 영향
        - 등장 빈도 수에 따라 자동 정렬
    - wv.key_to_index
        - 단어 -> 인덱스로 매칭
        - 특정 단어가 인덱스 몇에 위치하는가
    - copus_total_word
        - 전체 학습이 된 단어의 개수
    - epochs
        - 학습 epoch 수
    - vector_size
        - 벡터 차원의 수
- 메서드
    - wv.most_similar( word, topn = 10 )
        - 특정 단어와 유사한 단어를 출력
        - topn은 유사한 단어의 개수 지정
    - wv.similarity(word1, word2)
        - 두 단어 간의 코사인 유사도
    - wv.get_vector( word )
        - 특정 단어의 벡터를 반환
    - train()
        - 추가 데이터로 학습
    - save()
        - 학습된 모델을 저장
    - Word2Vec.load()
        - 저장되어있는 모델을 로드

In [1]:
# !pip install gensim

In [2]:
from gensim.models import Word2Vec

In [3]:
from sklearn.svm import SVC
from konlpy.tag import Komoran

In [ ]:
docs = [
    '오늘 날씨가 좋다 여행 가고 싶다', 
    '기온이 너무 올라서 아무것도 하기 싫다', 
    '수업이 너무 지루하고 졸리다', 
    '음식이 너무 맛이 없고 서비스도 별로다',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]
target = [1, 0, 0, 0, 1]

In [5]:
# Word2Vec은 토큰화 된 데이터가 필요
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']

tokens = []

for doc in docs:
    words = []
    for word, pos in komoran.pos(doc):
        if pos in allow_pos:
            words.append(word)
    tokens.append(words)

tokens

[['오늘', '날씨', '좋', '여행', '가'],
 ['기온', '너무', '오르', '아무것', '하', '싫'],
 ['수업', '너무', '졸리'],
 ['음식', '너무', '맛', '없', '서비스', '별로', '다'],
 ['영화', '너무', '재미있', '시간', '가', '모르']]

In [6]:
# Word2Vec을 이용하여 학습 (Skip-gram 방식)
w2v = Word2Vec(
    sentences = tokens, 
    vector_size=100, 
    window = 5, 
    min_count=1, 
    sg = 1, 
    epochs = 100, 
    seed = 42, 
    workers=2
)

In [7]:
# Word2Vec에서 wv 속성은 객체(class)로 반환 -> 자주 사용 되는 객체임으로 변수에 저장 
wv = w2v.wv

In [8]:
# wv에 특정 단어를 입력하면 벡터 출력
wv['여행']

array([ 0.00067393,  0.00060227,  0.0044239 , -0.00508487, -0.00337584,
       -0.00506212, -0.00414549, -0.00675128, -0.00933806,  0.00877077,
       -0.00856608,  0.00785082, -0.00989675,  0.00562794, -0.00337851,
        0.00042697, -0.00436719, -0.00029853, -0.00101309,  0.00058505,
       -0.00219302,  0.00076084,  0.00650614, -0.00127701, -0.00268797,
       -0.0073859 , -0.002808  , -0.00742177, -0.00639447,  0.00907987,
       -0.00822103, -0.00041097,  0.00665506,  0.00907863, -0.00704446,
       -0.00677937, -0.00250248,  0.00093587,  0.00914785, -0.00575276,
       -0.00664936, -0.00490768,  0.00760921, -0.00947477, -0.00244979,
       -0.00766411, -0.00686497,  0.00841504, -0.00982844, -0.00361847,
        0.00834756,  0.00207182, -0.0086334 , -0.000671  ,  0.00942244,
       -0.00202545,  0.00060924,  0.00068134,  0.00323355, -0.00620352,
        0.00690333,  0.00982491, -0.00727772,  0.00644986,  0.00644239,
        0.00483223, -0.00252508, -0.00059654,  0.00679188, -0.00

In [9]:
# 유사한 단어 찾기 
wv.most_similar('음식', topn = 3)

[('아무것', 0.11501993983983994),
 ('시간', 0.10833000391721725),
 ('영화', 0.07739505916833878)]

In [10]:
# 두 단어의 코사인 유사도를 확인 
wv.similarity('여행', '음식')

np.float32(-0.07214948)

In [11]:
len(wv.index_to_key)

23

In [12]:
import numpy as np

In [21]:
tokens[0]

['오늘', '날씨', '좋', '여행', '가']

In [23]:
# tokens의 단어들 중 w2v의 index_to_key에 존재하는 데이터의 단위 벡터를 확인
# vectors --> docs의 문장들을 벡터화한 리스트
vectors = []

for token in tokens:
    for word in token:
        # print(word)
        vec = []
        ## 문제점 -> token의 각 원소를 word에 대입하여 반복 실행하면서 매번 초기화 -> 마지막 단어의 벡터값만 vec에 대입
        if word in wv.index_to_key:
            # print(word)
            # tokens 데이터에서 단어가 w2v의 학습 단어에 포함되어있을때
            # 해당 단어의 벡터 값을 vec에 추가 
            vec.append(wv[word])
            # print(wv[word].shape)
    print(np.array(vec).shape)
    
    vectors.append( np.mean(vec, axis=0) )
    #     break
    # break    
vectors

(1, 100)
(1, 100)
(1, 100)
(1, 100)
(1, 100)


[array([ 6.61196932e-03, -5.97203663e-03,  6.20755740e-03, -1.01569984e-02,
         5.81729924e-03,  5.82032464e-03,  5.37980953e-03,  3.43962596e-03,
        -7.45705271e-04,  3.93294170e-03, -4.53955680e-03,  5.79827232e-03,
         1.24404463e-03, -6.49010646e-04,  1.91101819e-04,  1.49355934e-03,
        -9.22701601e-03, -7.47801876e-03, -5.23453392e-03, -7.78159499e-03,
        -1.33021956e-03,  3.38690239e-03,  3.08227679e-03, -5.28181030e-04,
         7.35988701e-03,  1.35731418e-03, -8.45187623e-03,  5.48181403e-03,
         1.37117703e-03,  2.82177934e-03,  1.30239152e-03,  1.03808625e-03,
        -8.32226500e-03,  1.25514588e-03,  5.94688905e-03, -4.15101787e-03,
         2.18014698e-03, -9.63387452e-03, -3.15451762e-03, -1.10267638e-03,
         9.83247254e-03, -5.71849896e-03, -4.23770538e-03, -1.93289400e-03,
         9.76065826e-03,  7.08405301e-03, -9.39494371e-03, -5.05532976e-03,
         6.76281378e-03, -8.90623126e-03,  7.19558308e-03, -4.65110317e-03,
         8.1

In [14]:
np.array(vectors).shape

(5, 100)

In [15]:
svc = SVC(random_state=42)

In [16]:
target

[1, 0, 0, 0, 1]

In [17]:
svc.fit(np.array(vectors), target)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [19]:
svc.predict(vectors)

array([1, 0, 0, 0, 1])